# Lecture 2.7 — Forcing and Controlling Tool Choice

**Section 02 — Agents: Configuration & Behaviour**  
**Course: OpenAI Agents SDK — Complete Course**

---

In this notebook we explore `ModelSettings.tool_choice` and `Agent.reset_tool_choice` — the two levers that control whether and which tools an agent calls during a run.

By the end of this notebook you will have seen all four `tool_choice` values in action (`"auto"`, `"required"`, `"none"`, and a specific function name), and you will understand the infinite-loop risk that comes with `"required"` and how `reset_tool_choice=True` (the default) prevents it.

## Cell 1 — Install the OpenAI Agents SDK

📌 **Notebook update notice:** this lecture's video and markdown reference `openai-agents==0.17.4` as the pinned version. Since recording, a downstream dependency change (`openai>=2.45.0`, released July 9, 2026) broke `openai-agents` versions below 0.18.1 — `Runner.run()` will fail on the version stated in the video. This notebook has been updated to pin `openai-agents==0.18.3`, which fixes the issue without changing any of the code or concepts taught in the lecture. Please use the version pinned below, not the one mentioned in the recording.

This cell installs the `openai-agents` package, pinned to version **0.18.3** for reproducibility. Pinning ensures that every step in this notebook runs against the same SDK behaviour, regardless of when you open it.

If you prefer to use a different version:
- To install the latest release: `pip install openai-agents`
- To use a specific version: replace `0.18.3` below with your preferred version number.

If the package is already installed in your current session, pip will confirm it is present and move on — you do not need to restart the kernel.

In [ ]:
# Pinned for reproducibility. Updated after recording — see the
# notice above. Originally pinned to 0.17.4 as stated in the
# video; updated to 0.18.3 to fix a breaking change introduced
# by openai>=2.45.0 (July 9, 2026).
# To use the latest version instead, run: pip install openai-agents
!pip install openai-agents==0.18.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 843.0/843.0 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 9.2 MB/s eta 0:00:00


## Cell 2 — API Key Setup (Google Colab Secrets)

This cell retrieves your OpenAI API key from Google Colab's Secrets store and writes it to the `OPENAI_API_KEY` environment variable, which the SDK reads automatically.

**Step-by-step instructions for Colab:**
1. Click the **key icon** (🔑) in the left sidebar to open the Secrets panel.
2. Click **+ Add new secret**.
3. Set the **Name** to `OPENAI_API_KEY`.
4. Paste your OpenAI API key as the **Value**.
5. Toggle **Notebook access** to ON for this notebook.
6. Run this cell.

> **Running locally?** Skip this cell and instead set the environment variable in your terminal before launching Jupyter:
> ```bash
> export OPENAI_API_KEY="your-key-here"
> ```

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3 — Model Name Variable

We declare a single `MODEL_NAME` variable here and use it throughout the notebook in every `Agent` definition. This means you can change the model for the entire notebook in one place — just update this cell and re-run.

The default is `"gpt-5.4-mini"`, a fast and cost-effective GPT-5 model. To use a different model, replace the string below with any model name from the OpenAI models page.

See the latest available models at: https://platform.openai.com/docs/models

In [ ]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 4 — Imports

Here we import everything needed for this notebook:

| Import | Source | Purpose |
|---|---|---|
| `Reasoning` | `openai.types.shared` | Controls reasoning effort on GPT-5 models. We pass `effort="none"` to minimise latency and cost during demos. |
| `Agent` | `agents` | The core agent class — wraps an LLM with instructions, tools, and configuration. |
| `ModelSettings` | `agents` | Holds model-level parameters including `tool_choice`. |
| `Runner` | `agents` | Executes agent runs. We always use `await Runner.run()` in notebooks (never `run_sync()`). |
| `function_tool` | `agents` | Decorator that converts a Python function into a tool the agent can call. |

> **Note on `function_tool`:** We use `@function_tool` here purely as a prop — two simple stub functions — to demonstrate `tool_choice` behaviour. The full `@function_tool` API (schema generation, type annotations, docstring parsing, context access, error handling, timeouts, etc.) is covered in depth in **Section 3**.

In [ ]:
from openai.types.shared import Reasoning
from agents import Agent, ModelSettings, Runner, function_tool

## Cell 5 — Why `tool_choice` Matters

### The problem: giving an agent tools doesn't mean it will use them

When you register tools on an agent, the model receives their schemas in the request — but it decides on its own whether to call any of them. By default the model uses `"auto"` mode: it calls a tool when it thinks it's useful, and answers directly when it thinks it isn't.

This is usually the right behaviour. But sometimes you need to override it:
- You want to **guarantee** a tool is called (e.g., a pipeline step where the tool call is the whole point).
- You want to **prevent** any tool from being called (e.g., a fallback path where you want a text answer, not tool overhead).
- You want to **force a specific tool** (e.g., a deterministic workflow where the model should not choose between tools).

### `ModelSettings.tool_choice` — the lever

`tool_choice` is a parameter on `ModelSettings`. It accepts four kinds of values:

| Value | Behaviour |
|---|---|
| `None` (default) | SDK omits `tool_choice` from the request. The model uses its provider default (typically `"auto"`). |
| `"auto"` | Model decides whether to call a tool and which one. This is the correct default for most agents. |
| `"required"` | Model **must** call at least one tool. It intelligently chooses which one. |
| `"none"` | Model **must not** call any tool, even if tools are registered. Falls back to a text response. |
| A specific function name (e.g. `"get_weather"`) | Model **must** call that exact tool. |

### The infinite-loop risk with `"required"`

There is a subtle but important danger when combining `tool_choice="required"` with the default `tool_use_behavior="run_llm_again"`:

1. Model is forced to call a tool.
2. Tool runs, result is sent back to the model.
3. Model is still forced to call a tool (because `tool_choice` is still `"required"`).
4. Repeat until `MaxTurnsExceeded`.

The SDK prevents this with `reset_tool_choice=True` (the default on every `Agent`). After the first tool call, the SDK automatically resets `tool_choice` to `"auto"` for subsequent turns in the same run. This means the model can freely produce a final text response after using the tool once.

We will see this in action in Cells 8 and 11.

## Cell 6 — Define Two Simple Tools

These two functions are deliberately minimal — they are props for demonstrating `tool_choice` behaviour, not realistic implementations. Their docstrings become the tool descriptions that the model sees, and their type annotations (`city: str`) generate the parameter schema automatically.

**What `@function_tool` does here (brief preview):**
- Wraps the function so the agent can call it.
- Uses the docstring as the tool description.
- Uses the `city: str` annotation to generate a JSON schema with a required `city` string parameter.
- Returns the function's return value to the model as the tool result.

The full `@function_tool` API — including advanced schema generation, `RunContextWrapper` context access, custom error handling, timeouts, `name_override`, `description_override`, and more — is covered in **Section 3**.

In [ ]:
@function_tool
def get_weather(city: str) -> str:
    """Returns the current weather for a city."""
    print("[TOOL CALL] get_weather called")
    return f"The weather in {city} is sunny and 24°C."


@function_tool
def get_time(city: str) -> str:
    """Returns the current local time for a city."""
    print("[TOOL CALL] get_time called")
    return f"The current time in {city} is 14:35."

## Cell 7 — `tool_choice="auto"` — The Default

With `tool_choice="auto"` the model decides on its own whether to call a tool. For a geography question (`"What is the capital of France?"`) it has no reason to call either tool, so it answers directly from its own knowledge. For a weather question (`"What is the weather in Tokyo?"`) it correctly identifies `get_weather` as the right tool and calls it.

This is the correct setting for most agents. The model is smart enough to know when tools are useful — you don't need to force it.

We also set `reasoning=Reasoning(effort="none")` and `verbosity="low"` to keep latency and cost low for demos. These are appropriate for GPT-5 models; for non-GPT-5 models, omit the `reasoning` parameter.

In [ ]:
agent_auto = Agent(
    name="Auto Agent",
    instructions=(
        "You are a helpful assistant. "
        "Answer the user's question directly or use "
        "tools if needed."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
        tool_choice="auto",
    ),
    tools=[get_weather, get_time],
)

# Geography question — no tool needed
result_no_tool = await Runner.run(
    agent_auto,
    "What is the capital of France?",
)
print("Auto (no tool needed):", result_no_tool.final_output)

# Weather question — tool will be called
result_with_tool = await Runner.run(
    agent_auto,
    "What is the weather in Tokyo?",
)
print("Auto (tool used):", result_with_tool.final_output)

Auto (no tool needed): Paris
[TOOL CALL] get_weather called
Auto (tool used): Tokyo: sunny, 24°C.


## Cell 8 — `tool_choice="required"` — Force a Tool Call

With `tool_choice="required"` the model **must** call at least one tool, regardless of whether the question warrants it. Here we ask about the capital of France — a question that needs no tool. The model is still forced to pick one, so it picks the most plausible-looking tool from what's available (likely `get_weather` or `get_time`) and calls it.

**Key point — why this doesn't loop:**  
`reset_tool_choice=True` is the default on every `Agent`. After the model makes its forced tool call, the SDK resets `tool_choice` to `"auto"` for subsequent turns. This allows the model to produce a final text response without being forced into another tool call.

**When to use `"required"`:** Deterministic pipelines where your workflow depends on the agent always producing a tool call — for example, a pipeline step where the tool call *is* the point of the step and a text-only response would break downstream logic.

> **Rule F4 reminder:** Always use `await Runner.run()` in notebooks. `run_sync()` raises `RuntimeError` in Jupyter/Colab environments that already have an event loop running.

In [ ]:
agent_required = Agent(
    name="Required Agent",
    instructions="You are a helpful assistant.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
        tool_choice="required",
    ),
    tools=[get_weather, get_time],
)

result = await Runner.run(
    agent_required,
    "What is the capital of France?",
)
print("Required:", result.final_output)

[TOOL CALL] get_time called
Required: Paris.


## Cell 9 — `tool_choice="none"` — Prevent Any Tool Call

With `tool_choice="none"` the model is prevented from calling any tool, even though both tools are registered. We ask about the weather in Tokyo — exactly the question `get_weather` is designed for. But with `"none"`, the model must fall back to generating a text response from its own knowledge instead.

**When to use `"none"`:**  
This is a useful pattern for conditionally disabling tools at runtime without removing them from the agent definition. For example:
- A fallback path where you want a fast text answer without any tool overhead.
- A scenario where tool access should be gated by some runtime condition.
- Testing what the agent knows without tool assistance.

The agent object itself doesn't change — only the `tool_choice` setting on `ModelSettings` is different.

In [ ]:
agent_none = Agent(
    name="No Tool Agent",
    instructions="You are a helpful assistant.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
        tool_choice="none",
    ),
    tools=[get_weather, get_time],
)

result = await Runner.run(
    agent_none,
    "What is the weather in Tokyo?",
)
print("No tools:", result.final_output)

No tools: I can’t check live weather right now. If you want, I can still help you find a reliable source or give a general Tokyo weather overview by season.


## Cell 10 — Forcing a Specific Tool by Name

You can pass a function name string to `tool_choice` to force the model to call that specific tool first. Here we force `"get_weather"` even though the question is about time.

**What actually happens with the default settings:**  
Forcing a specific tool name only guarantees that tool is called *first*. After it runs, `reset_tool_choice=True` (the default) kicks in and resets `tool_choice` to `"auto"`. The model is then free to call additional tools if it decides to. In this example the model calls `get_weather` as forced, then decides on its own to also call `get_time` since the question was about time — and uses the `get_time` result for the final answer.

**To force exactly one tool and stop immediately, combine with `tool_use_behavior="stop_on_first_tool"`:**  
This stops the run the moment the first tool call completes. The tool's raw output becomes the final result directly — the model never gets another turn. This is the truly deterministic pattern.

This notebook shows **both versions** side by side so you can see the difference clearly.

**Important constraint (verified from SDK source):**  
On the OpenAI Responses API, named `tool_choice` must target a **top-level callable function tool**. You cannot target:
- A namespace wrapper created by `tool_namespace()`
- A bare inner name from `tool_namespace()`
- A deferred-only tool (one with `defer_loading=True`)

For those cases, prefer `"auto"` or `"required"`.

In [ ]:
# VERSION 1: Default behaviour — tool_choice forces get_weather first,
# but reset_tool_choice=True allows the model to call more tools after.
agent_specific_default = Agent(
    name="Weather Only Agent (default)",
    instructions="You are a helpful assistant.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
        tool_choice="get_weather",
    ),
    tools=[get_weather, get_time],
    # reset_tool_choice=True is the default — after get_weather runs,
    # tool_choice resets to "auto" and the model may call more tools.
)

result_default = await Runner.run(
    agent_specific_default,
    "What time is it in Paris?",
)
print("Default (may call multiple tools):", result_default.final_output)


# VERSION 2: Truly deterministic — stop_on_first_tool halts the run
# immediately after get_weather runs. The model never gets another turn.
agent_specific_stop = Agent(
    name="Weather Only Agent (stop)",
    instructions="You are a helpful assistant.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
        tool_choice="get_weather",
    ),
    tools=[get_weather, get_time],
    tool_use_behavior="stop_on_first_tool",
)

result_stop = await Runner.run(
    agent_specific_stop,
    "What time is it in Paris?",
)
print("Stop on first tool (raw tool output):", result_stop.final_output)

[TOOL CALL] get_weather called
[TOOL CALL] get_time called
Default (may call multiple tools): The current time in Paris is 14:35.
[TOOL CALL] get_weather called
Stop on first tool (raw tool output): The weather in Paris is sunny and 24°C.


## Cell 11 — The Infinite Loop Problem and `reset_tool_choice`

This cell makes the infinite-loop risk concrete and shows the safe default.

### The dangerous pattern

The following configuration creates an infinite loop — **do not run it**:

- `tool_choice="required"` — model must call a tool every turn.
- `tool_use_behavior="run_llm_again"` — the default; after a tool call, the result is sent back to the model for another turn.
- `reset_tool_choice=False` — `tool_choice` is **not** reset after the first tool call.

Result: tool called → result sent to model → model forced to call tool again → repeat → `MaxTurnsExceeded`.

### The safe default

`reset_tool_choice=True` is the default on every `Agent` and you should almost never change it. After the first tool call, the SDK resets `tool_choice` to `"auto"`. The model can then produce a final text response without being forced into another tool call.

We show `reset_tool_choice=True` explicitly here to make it visible — it is already the default and you do not need to set it manually.

**When is `reset_tool_choice=False` acceptable?** Only when you have a stopping condition elsewhere — for example, `tool_use_behavior="stop_on_first_tool"`, which stops the run immediately after the first tool call and uses its output as the final result. `tool_use_behavior` is covered fully in **Lecture 2.8**.

In [ ]:
# DANGEROUS — DO NOT RUN:
# agent_loop = Agent(
#     name="Loop Agent",
#     model=MODEL_NAME,
#     model_settings=ModelSettings(tool_choice="required"),
#     tool_use_behavior="run_llm_again",  # default
#     reset_tool_choice=False,            # dangerous!
#     tools=[get_weather],
# )
# This loops: tool called → result to LLM → LLM forced
# to call tool again → repeat until MaxTurnsExceeded


# SAFE PATTERN — reset_tool_choice=True (default, shown explicitly)
agent_safe = Agent(
    name="Safe Required Agent",
    instructions="You are a helpful assistant.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
        tool_choice="required",
    ),
    tools=[get_weather, get_time],
    reset_tool_choice=True,  # default — shown explicitly for clarity
)

result = await Runner.run(
    agent_safe,
    "What is the weather in London?",
)
print("Safe:", result.final_output)
print("Items generated:", len(result.new_items))

[TOOL CALL] get_weather called
Safe: London: sunny, 24°C.
Items generated: 3


## Cell 12 — Safe and Unsafe Combinations (Summary)

Use this table as a quick reference when combining `tool_choice`, `tool_use_behavior`, and `reset_tool_choice`.

| `tool_choice` | `tool_use_behavior` | `reset_tool_choice` | Safe? |
|---|---|---|---|
| `"required"` | `"run_llm_again"` | `True` (default) | ✅ Safe — resets after first call |
| `"required"` | `"stop_on_first_tool"` | Either | ✅ Safe — stops after first tool |
| `"required"` | `"run_llm_again"` | `False` | ❌ **INFINITE LOOP** |
| `"auto"` | `"run_llm_again"` | Either | ✅ Always safe |
| `"none"` | Any | Either | ✅ Safe — no tool called |
| Specific name | `"stop_on_first_tool"` | Either | ✅ Safe and deterministic |

The key rule: **if you use `tool_choice="required"` with `tool_use_behavior="run_llm_again"` (the default), always leave `reset_tool_choice=True` (the default).** Only set `reset_tool_choice=False` if you have an explicit stopping condition elsewhere.

`tool_use_behavior` is covered fully in **Lecture 2.8**.